# Construction-PPE -> YOLO26 (presence classes only)

RunPod runbook: download -> strip `no_*` / `none` classes -> train -> evaluate.

Everything persistent lives under `/workspace` (the RunPod network volume). Anything outside it
is wiped when the pod restarts.

**Baseline to beat** (earlier 11-class run, presence classes only, test split):
macro mAP50 ~= 0.836, mAP50-95 ~= 0.433.

## 0. Install & verify

Run the install, then **Kernel -> Restart Kernel** before anything else. An in-place upgrade of
an already-imported package does not take effect otherwise.

In [ ]:
%pip install -q -U ultralytics
# ---> now do Kernel -> Restart Kernel, then continue with the next cell

In [ ]:
import torch, ultralytics
ultralytics.checks()
print("ultralytics", ultralytics.__version__)
print("torch      ", torch.__version__)
print("gpu        ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

## 1. Check what is already on the volume

If you re-attached the same network volume, the old (flat, broken) dataset may still be there.

In [ ]:
from pathlib import Path

root = Path("/workspace/datasets")
if root.exists():
    for p in sorted(root.iterdir()):
        print(("dir  " if p.is_dir() else "file ") + p.name)
else:
    print("/workspace/datasets does not exist yet -- clean slate")

### 1b. Optional clean slate

Only run this if the listing above shows the old flat layout (`images/`, `labels/`, `data.yaml`
sitting loose in `/workspace/datasets`). It deletes that whole directory.

In [ ]:
# DESTRUCTIVE -- only run if the cell above showed the old broken layout.
import shutil
from pathlib import Path

root = Path("/workspace/datasets")
if root.exists():
    print("deleting:", sorted(p.name for p in root.iterdir()))
    shutil.rmtree(root)
    print("done")

## 2. Download + extract

The zip is **flat** -- `images/`, `labels/`, `data.yaml`, `LICENSE` land directly in the
extraction directory. So extract into its own folder, not into `/workspace/datasets` root.

In [ ]:
import urllib.request, zipfile
from pathlib import Path

URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip"
RAW = Path("/workspace/datasets/construction-ppe")
ZIP = Path("/workspace/datasets/construction-ppe.zip")

RAW.mkdir(parents=True, exist_ok=True)
if not ZIP.exists():
    print("downloading ...")
    urllib.request.urlretrieve(URL, ZIP)      # follows the GitHub redirect

with zipfile.ZipFile(ZIP) as z:
    z.extractall(RAW)                         # flat zip -> contents land inside RAW

print(sorted(p.name for p in RAW.iterdir()))

## 3. Inspect classes and per-split counts

Confirms the class list and the id of every class before anything gets remapped.

In [ ]:
import yaml
from collections import Counter
from pathlib import Path

RAW = Path("/workspace/datasets/construction-ppe")
cfg = yaml.safe_load((RAW / "data.yaml").read_text())

names = cfg["names"]
names = [names[i] for i in sorted(names)] if isinstance(names, dict) else list(names)
print(f"{len(names)} classes:")
for i, n in enumerate(names):
    print(f"  {i:>2}  {n}")
print("\nyaml (minus names):", {k: v for k, v in cfg.items() if k != "names"})

for split in ("train", "val", "test"):
    d = RAW / "labels" / split
    if not d.exists():
        print(f"\n{split}: MISSING")
        continue
    counts, files = Counter(), 0
    for f in d.glob("*.txt"):
        files += 1
        counts.update(int(l.split()[0]) for l in f.read_text().splitlines() if l.strip())
    print(f"\n{split}: {files} label files, {sum(counts.values())} boxes")
    for i in sorted(counts, key=lambda k: -counts[k]):
        print(f"   {i:>2} {names[i]:<12} {counts[i]:>6}")

## 4. Build the presence-only dataset

Keeps `helmet, gloves, vest, boots, goggles, Person`; drops `no_helmet, no_goggle, no_gloves,
no_boots, none`. Class ids are remapped to a contiguous `0..5` -- this is the part that must be
right, since a leftover id >= nc is what caused the CUDA `index out of bounds` crash.

- Images are **symlinked**, not copied, so there is no second copy of the dataset on the volume.
  If you later move the originals the links break -- use `shutil.copy2` instead if that matters.
- Images whose labels all got dropped are kept as **background images** with an empty `.txt`.
  That is deliberate: they teach the model not to fire on unequipped workers, which is exactly
  the failure mode you would otherwise introduce by deleting the absence labels. If the reported
  background share goes much above ~20% of a split, reconsider.

In [ ]:
from pathlib import Path
import yaml

SRC  = Path("/workspace/datasets/construction-ppe")
DST  = Path("/workspace/datasets/ppe-presence")
KEEP = ["helmet", "gloves", "vest", "boots", "goggles", "Person"]
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

names = yaml.safe_load((SRC / "data.yaml").read_text())["names"]
names = [names[i] for i in sorted(names)] if isinstance(names, dict) else list(names)

missing = [k for k in KEEP if k not in names]
assert not missing, f"not in dataset: {missing}\navailable: {names}"

remap = {names.index(k): i for i, k in enumerate(KEEP)}     # old id -> new id
print("remap:", {names[o]: f"{o} -> {n}" for o, n in remap.items()})
print("dropping:", [n for i, n in enumerate(names) if i not in remap], "\n")

splits = {}
for split in ("train", "val", "test"):
    simg, slbl = SRC / "images" / split, SRC / "labels" / split
    if not simg.exists():
        continue
    dimg, dlbl = DST / "images" / split, DST / "labels" / split
    dimg.mkdir(parents=True, exist_ok=True)
    dlbl.mkdir(parents=True, exist_ok=True)

    kept = dropped = background = n_img = 0
    for img in sorted(simg.iterdir()):
        if img.suffix.lower() not in IMG_EXT:
            continue
        n_img += 1
        link = dimg / img.name
        if not link.exists():
            link.symlink_to(img)

        out, lbl = [], slbl / f"{img.stem}.txt"
        if lbl.exists():
            for ln in lbl.read_text().splitlines():
                if not ln.strip():
                    continue
                parts = ln.split()
                cid = int(parts[0])
                if cid in remap:
                    out.append(" ".join([str(remap[cid])] + parts[1:]))
                    kept += 1
                else:
                    dropped += 1
        (dlbl / f"{img.stem}.txt").write_text("\n".join(out) + ("\n" if out else ""))
        background += not out

    splits[split] = f"images/{split}"
    print(f"{split:<5} {n_img:>5} imgs | kept {kept:>5} | dropped {dropped:>5} | "
          f"background-only {background:>4} ({background / max(n_img, 1):.0%})")

cfg = {"path": str(DST), **splits, "names": {i: n for i, n in enumerate(KEEP)}}
(DST / "data.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))
print("\n" + (DST / "data.yaml").read_text())

## 5. Verification gate

Do not skip this. It proves every label id is `< nc`, and clears stale `.cache` files.
Ultralytics caches parsed labels next to each split; a cache written under a *different* class
count is the other half of that CUDA crash.

In [ ]:
from pathlib import Path
import yaml

DST = Path("/workspace/datasets/ppe-presence")
cfg = yaml.safe_load((DST / "data.yaml").read_text())
nc  = len(cfg["names"])

clean = True
for split in ("train", "val", "test"):
    d = DST / "labels" / split
    if not d.exists():
        continue
    mx, bad = -1, []
    for f in d.glob("*.txt"):
        for ln in f.read_text().splitlines():
            if not ln.strip():
                continue
            cid = int(ln.split()[0])
            mx = max(mx, cid)
            if cid >= nc:
                bad.append((f.name, cid))
    clean &= not bad
    print(f"{split:<5} max id {mx} / nc {nc}   " +
          ("OK" if not bad else f"MISMATCH {bad[:5]}"))

for c in DST.rglob("*.cache"):
    c.unlink()
    print("removed stale cache:", c)

assert clean, "label id >= nc -- fix this before training or CUDA will assert"
print("\ndataset is self-consistent -- safe to train")

## 6. Train

- `project` points at `/workspace` so weights survive a pod restart.
- `cache="ram"` sidesteps the `Slow image access detected` warning -- `/workspace` is a network
  volume (~15 MB/s), which bottlenecks a 4090. This dataset is small enough to sit in RAM.
  If memory becomes a problem, drop to `cache=False`.
- `batch=32` explicitly rather than `-1`: reproducible across runs, and skips the AutoBatch
  probe. A 4090 handles `yolo26s` at 640px comfortably here.
- `patience=30` early-stops once val mAP stops improving, so 150 is an upper bound, not a
  commitment.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26s.pt")          # n / s / m / l / x
results = model.train(
    data="/workspace/datasets/ppe-presence/data.yaml",
    epochs=150,
    imgsz=640,
    batch=32,
    cache="ram",
    device=0,
    workers=8,
    seed=0,
    patience=30,
    plots=True,
    project="/workspace/runs",
    name="ppe-presence-s",
)
print("weights ->", results.save_dir)

## 7. Evaluate on the test split

Called inline -- no `argparse`, which is what produced the `unrecognized arguments: -f`
SystemExit when a script body was pasted straight into a cell.

`conf=0.001` is deliberate: mAP is defined over the full precision-recall curve, so evaluation
needs the low-confidence detections included. For numbers that reflect production behaviour,
re-run with `conf=0.25` -- those P/R figures will look quite different, and that is expected.

In [ ]:
from ultralytics import YOLO

WEIGHTS = "/workspace/runs/ppe-presence-s/weights/best.pt"

model = YOLO(WEIGHTS)
m = model.val(
    data="/workspace/datasets/ppe-presence/data.yaml",
    split="test",
    imgsz=640, batch=16,
    conf=0.001,
    iou=0.6,
    plots=True,
    project="/workspace/runs", name="eval-presence-test",
)

rows = [(model.names[ci], *m.box.class_result(i)) for i, ci in enumerate(m.box.ap_class_index)]
rows.sort(key=lambda r: r[4])                      # worst mAP50-95 first
w = max([len(r[0]) for r in rows] + [5])
head = f"{'class':<{w}}  {'P':>6}  {'R':>6}  {'mAP50':>7}  {'mAP50-95':>9}"
print("\n" + head)
print("-" * len(head))
for n, p, r, ap50, ap in rows:
    print(f"{n:<{w}}  {p:>6.3f}  {r:>6.3f}  {ap50:>7.3f}  {ap:>9.3f}")
print("-" * len(head))
print(f"{'all':<{w}}  {m.box.mp:>6.3f}  {m.box.mr:>6.3f}  {m.box.map50:>7.3f}  {m.box.map:>9.3f}")

s = m.speed
total = sum(s.values())
print(f"\nspeed  : {s['preprocess']:.1f} pre + {s['inference']:.1f} inf + "
      f"{s['postprocess']:.1f} post = {total:.1f} ms/img  ({1000 / total:.0f} FPS)")
print(f"plots  : {m.save_dir}   (confusion matrix, PR / F1 curves)")
print("\nbaseline to beat (11-class run, presence classes only): mAP50 0.836, mAP50-95 0.433")

## 8. Optional: YOLO26 NMS-free head

YOLO26 has a one-to-one head you opt into with `nms=False`. Worth measuring once before
committing to an end-to-end export -- postprocess time drops to near zero, usually at a small
accuracy cost.

In [ ]:
mf = model.val(
    data="/workspace/datasets/ppe-presence/data.yaml",
    split="test", imgsz=640, batch=16, conf=0.001, iou=0.6,
    nms=False,
    plots=True,
    project="/workspace/runs", name="eval-presence-test-nmsfree",
)
print(f"nms-free : mAP50 {mf.box.map50:.3f}  mAP50-95 {mf.box.map:.3f}  "
      f"post {mf.speed['postprocess']:.2f} ms")
print(f"with nms : mAP50 {m.box.map50:.3f}  mAP50-95 {m.box.map:.3f}  "
      f"post {m.speed['postprocess']:.2f} ms")

## 9. Where things live

```
/workspace/datasets/construction-ppe/     raw download (11 classes, untouched)
/workspace/datasets/ppe-presence/         6-class dataset, images symlinked
/workspace/runs/ppe-presence-s/weights/   best.pt, last.pt
/workspace/runs/eval-presence-test/       confusion matrix, PR / F1 curves
```

Once training is confirmed working, `/workspace/datasets/construction-ppe.zip` can be deleted.

**If the numbers disappoint.** The earlier run showed a wide mAP50 vs mAP50-95 gap (0.568 vs
0.270) -- boxes found but fitted loosely, driven by the small classes (gloves, goggles). Levers
in order of payoff: `imgsz=960`, then `yolo26m`, then data work. Check the confusion matrix in
the eval run directory first.